# 03 · Train baseline

Fits the two reference conditions: the n-gram, tuned on validation, and the context-32 transformer (seed 0). The test split is not used here.

In [ ]:
%run 02_model.ipynb

## Reference 1: n-gram

Order (1–8) and discount *d* are chosen by validation BPC. The model is deterministic, so it needs only one run.

In [ ]:
ngram = NgramModel(train_text, max_order=8)

ngram_grid = []
for discount in [0.5, 0.75, 0.9]:
    for order, bpc in enumerate(ngram_bpc_by_order(ngram, val_text, discount), start=1):
        ngram_grid.append({"order": order, "discount": discount, "val_bpc": float(bpc)})

best_ngram = min(ngram_grid, key=lambda row: row["val_bpc"])
(RESULTS_DIR / "ngram.json").write_text(json.dumps({"best": best_ngram, "grid": ngram_grid}, indent=2))
best_ngram

## Training loop

AdamW with a cosine learning-rate schedule from 1e-3 down to 1e-4, gradient clipping at 1.0, and fp16 mixed precision on GPU. Every `eval_interval` steps the loop:

- estimates the validation loss on a fixed set of validation batches;
- keeps the checkpoint with the lowest validation loss as `best.pt`;
- saves the full training state to `last.pt`, so a run that disconnects picks up where it stopped.

In [ ]:
import time
from dataclasses import asdict


def get_batch(data, context_len, batch_size, generator):
    starts = torch.randint(len(data) - context_len, (batch_size,), generator=generator)
    positions = starts[:, None] + torch.arange(context_len)
    return data[positions].to(DEVICE), data[positions + 1].to(DEVICE)


def loss_on(model, inputs, targets):
    with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
        logits = model(inputs)
    return F.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())


@torch.no_grad()
def validation_loss(model, config):
    model.eval()
    generator = torch.Generator().manual_seed(0)
    losses = [loss_on(model, *get_batch(val_data, config.context_len, config.batch_size, generator)).item()
              for _ in range(config.eval_batches)]
    model.train()
    return sum(losses) / len(losses)


def train(config):
    run_dir = CHECKPOINT_DIR / config.run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"

    model = CharTransformer(config).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.max_steps, eta_min=config.min_lr)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    batch_generator = torch.Generator().manual_seed(config.seed)
    progress = {"step": 0, "best_step": 0, "best_val_loss": math.inf, "train_seconds": 0.0, "history": []}

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        batch_generator.set_state(checkpoint["batch_generator"])
        progress = checkpoint["progress"]
        print(f"Resuming {config.run_name} from step {progress['step']}")

    torch.manual_seed(config.seed + progress["step"])
    model.train()
    start_time, recent_losses = time.time(), []
    for step in range(progress["step"] + 1, config.max_steps + 1):
        loss = loss_on(model, *get_batch(train_data, config.context_len, config.batch_size, batch_generator))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        recent_losses.append(loss.item())

        if step % config.eval_interval == 0:
            val_loss = validation_loss(model, config)
            progress["history"].append({"step": step, "train_loss": sum(recent_losses) / len(recent_losses), "val_loss": val_loss})
            recent_losses = []
            if val_loss < progress["best_val_loss"]:
                progress.update(best_step=step, best_val_loss=val_loss)
                torch.save(model.state_dict(), best_path)
            progress["step"] = step
            progress["train_seconds"] += time.time() - start_time
            start_time = time.time()
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "batch_generator": batch_generator.get_state(),
                "progress": progress,
            }, last_path)
            print(f"step {step:5d} | train {progress['history'][-1]['train_loss']:.4f} | val {val_loss:.4f} "
                  f"| best {progress['best_val_loss']:.4f} @ {progress['best_step']}")

    run_record = {
        "config": asdict(config),
        "batch_size": config.batch_size,
        "num_params": model.num_params(),
        "device": torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu",
        "torch_version": torch.__version__,
        **progress,
    }
    (RUNS_DIR / f"{config.run_name}.json").write_text(json.dumps(run_record, indent=2))
    return run_record

## Reference 2: context-32 transformer, seed 0

In [ ]:
reference_run = train(Config(context_len=32, seed=0))
print(f"best validation loss {reference_run['best_val_loss']:.4f} at step {reference_run['best_step']}, "
      f"{reference_run['train_seconds'] / 60:.1f} min on {reference_run['device']}")